# 3. App Registrations and Managed Identities

AZ-500 expects you to configure OAuth flows, manage consent, and implement managed identities.
This builds on the [Azure Authentication lab](../../../08-enterprise/azure-authentication/).

## Before you run this notebook

1. Start the tiny mock Entra ID (built from the `08-enterprise/azure-authentication` lab):
   ```bash
   cd security-certs/az-500/01-identity-and-access
   docker compose up -d
   uv sync
   ```
2. In VS Code, pick the `.venv` kernel at the top-right. Reload the window if it's not listed.
3. Check the mock is reachable:
   ```bash
   curl http://localhost:9100/health
   ```
   You should see `{"status":"ok",...}`.

> **Heads-up:** The mock speaks just enough OAuth2 for this lab (`client_credentials`,
> `password` / ROPC, and on-behalf-of). Real Entra ID has many more features and you'd
> almost never enable ROPC in production.

## App registration checklist

| Setting | Where in the portal | Purpose |
|---------|---------------------|---------|
| **Display name** | Overview | Human-readable identifier |
| **Supported account types** | Authentication | Single tenant, multi-tenant, or personal |
| **Redirect URIs** | Authentication | Where tokens are sent after interactive sign-in |
| **Client secret or certificate** | Certificates & secrets | Prove identity for confidential clients |
| **API permissions** | API permissions | What resources the app can access |
| **Expose an API** | Expose an API | Scopes other apps can request |
| **App roles** | App roles | Application-level permissions (no user) |


In [ ]:
import httpx, json, base64

ENTRA = 'http://localhost:9100/contoso'
TOKEN_URL = f'{ENTRA}/oauth2/v2.0/token'

def decode(token: str) -> dict:
    # Quick-and-dirty JWT payload decoder. DO NOT use this to validate tokens in
    # real code. Your API library (FastAPI + python-jose, MSAL, etc.) must verify
    # the signature, issuer, audience and expiry before trusting anything.
    payload = token.split('.')[1]
    return json.loads(base64.urlsafe_b64decode(payload + '=' * (-len(payload) % 4)))

# --- 1. Delegated permission (user is present) ------------------------------
# The api-a application is acting ON BEHALF OF Alice. The token has an `scp`
# claim listing the delegated scopes she's consented to.
print('=== 1. Delegated permission (OAuth scope) ===')
r = httpx.post(TOKEN_URL, data={
    'grant_type': 'password',
    'client_id': 'api-a-client-id',
    'client_secret': 'api-a-secret-value',
    'username': 'alice@contoso.com',
    'password': 'alice-password',
    'scope': 'api://api-b/Files.Read',
})
r.raise_for_status()
delegated = decode(r.json()['access_token'])
print('Has "scp" claim (delegated), no "roles" claim.')
print(f'  scp (scopes): {delegated.get("scp")}')
print(f'  upn (user):   {delegated.get("upn")}')
print(f'  aud:          {delegated.get("aud")}')

# --- 2. Application permission (no user) ------------------------------------
# A daemon/cron calling api-b on its own behalf. The token has a `roles` claim
# listing the app roles the daemon was pre-assigned. There is no user.
print('\n=== 2. Application permission (app role) ===')
r = httpx.post(TOKEN_URL, data={
    'grant_type': 'client_credentials',
    'client_id': 'daemon-client-id',
    'client_secret': 'daemon-secret-value',
    'scope': 'api://api-b/.default',
})
r.raise_for_status()
app_only = decode(r.json()['access_token'])
print('Has "roles" claim (app-only), no user claims.')
print(f'  roles:        {app_only.get("roles")}')
print(f'  upn:          {app_only.get("upn", "(none - app-only token)")}')
print(f'  aud:          {app_only.get("aud")}')

# --- 3. The same call WITHOUT admin consent ---------------------------------
# reporting-daemon was never granted an app role on api-b. Note what happens:
# the token endpoint still returns 200 with a perfectly valid signed token. The
# failure surfaces at the RESOURCE, as a 403, not at the identity provider.
print('\n=== 3. Application permission with NO admin consent granted ===')
r = httpx.post(TOKEN_URL, data={
    'grant_type': 'client_credentials',
    'client_id': 'reporting-daemon-client-id',
    'client_secret': 'reporting-daemon-secret-value',
    'scope': 'api://api-b/.default',
})
r.raise_for_status()
unconsented = decode(r.json()['access_token'])
print(f'  HTTP status:  {r.status_code}  <- still a 200!')
print(f'  roles:        {unconsented.get("roles")}  <- empty: nothing was consented')
print('  api-b will reject this token with 403, and the app owner will swear')
print('  "but I got a token". Getting a token is not the same as being authorised.')

# The claim shapes ARE the lesson of this cell, so assert them.
assert 'scp' in delegated and 'roles' not in delegated, \
    'a delegated token carries scp (scopes) and no roles claim'
assert delegated['upn'] == 'alice@contoso.com', 'a delegated token identifies the user'
assert 'roles' in app_only and 'scp' not in app_only, \
    'an app-only token carries roles (app roles) and no scp claim'
assert 'upn' not in app_only, 'an app-only token has no user - there is nobody to name'
assert app_only['roles'] == ['Files.Read.All'], 'the daemon was granted this app role'
assert unconsented['roles'] == [], \
    'no admin consent means an EMPTY roles claim, not a failed token request'
assert delegated['aud'] == app_only['aud'] == 'api://api-b', \
    'both tokens are audienced at api-b - only the authorisation model differs'
print('\nToken shape invariants hold.')


## Bad to Best: handling the daemon's credentials

The daemon above authenticates with a **client secret** we pasted into code. That's exactly the anti-pattern managed identities exist to fix.


In [ ]:
approaches = [
    ('BAD',
     'Client secret hard-coded in source control.',
     'One `git clone` and an attacker has your app\'s identity. Rotation means a code change.'),
    ('OKAY',
     'Client secret in a Key Vault, fetched at startup.',
     'Better, but *something* still has to authenticate to Key Vault without a secret. Chicken and egg.'),
    ('GOOD',
     'Client certificate (private key in Key Vault, rotated automatically).',
     'Stronger than secrets, but you still manage rotation.'),
    ('BEST',
     'Managed identity (system- or user-assigned) on the hosting resource.',
     'Azure handles creation, rotation and retirement. There is no secret to leak.'),
    ('BEST (for CI/CD)',
     'Workload Identity Federation (OIDC) - e.g. GitHub Actions trusts Entra ID directly.',
     'No secret to rotate, no managed identity needed outside Azure.'),
]
for tag, setup, impact in approaches:
    print(f'{tag:<18}  {setup}')
    print(f'                    -> {impact}\n')


## OAuth consent

| Type | Who approves | When |
|------|-------------|------|
| **User consent** | The individual user, for their own access | Low-risk delegated permissions (e.g. `User.Read`) |
| **Admin consent** | A Privileged Role Administrator or Global Administrator, on behalf of the whole tenant | High-risk delegated permissions, or *any* application permission |
| **Pre-authorization** | The owner of the **API being called** | The API owner adds a client app + scope to its "Authorized client applications" list, so no consent prompt ever appears for that pair |

Application permissions (app roles) can **never** be user-consented — there is no user in
the flow to consent. They always require admin consent. And as the code cell above showed,
missing consent does not fail at the token endpoint: you get a valid token with an empty
`roles` claim, and the 403 arrives later at the resource.

### Managing consent

```bash
# Grant admin consent for an app.
az ad app permission admin-consent --id <app-id>

# See what's currently granted.
az ad app permission list --id <app-id>

# Restrict user consent to verified publishers and low-risk permissions.
az rest --method PATCH \
  --uri 'https://graph.microsoft.com/v1.0/policies/authorizationPolicy' \
  --body '{"defaultUserRolePermissions": {"permissionGrantPoliciesAssigned": ["managePermissionGrantsForSelf.microsoft-user-default-low"]}}'
```

### Exam tip: illicit consent grants

A common attack: an attacker tricks a user into consenting to an app that asks for `Mail.Read` or `Files.ReadWrite.All`. The attacker doesn't need the user's password - consent is enough. Defend by:

1. Restricting user consent to verified publishers.
2. Requiring admin consent for high-risk permissions.
3. Running the **Admin consent workflow** so users *request* consent instead of blindly approving.
4. Reviewing existing grants regularly.

---
## Managed identities - when to use which

| | System-assigned | User-assigned |
|-|----------------|---------------|
| **Lifecycle** | Created and **deleted with the resource** - delete the VM and the identity, its object ID and every role assignment made to it are gone | An **independent Azure resource** with its own lifecycle; it survives every workload you attach it to |
| **Sharing** | Exactly one resource, 1:1, forever | Attach the same identity to many resources |
| **Use case** | Single-purpose workload | Shared identity (e.g. many VMs hitting the same Key Vault); also the only option when the RBAC grant must exist *before* the resource does |
| **Deployment slots** | New identity per slot | Same identity across slots |
| **Re-created resource** | New object ID -> **all role assignments must be granted again** | Same object ID, nothing to redo |

That last row is the one that bites in production: rebuild a VM with a system-assigned
identity and every `az role assignment create` you ran against its principal ID is stale,
because the principal no longer exists. A user-assigned identity is the fix, and it is also
what lets you grant permissions in one pipeline stage and create the compute in a later one.

### Azure CLI reference


In [ ]:
cli = {
    'Enable system-assigned MI on a VM':
        'az vm identity assign -g rg-prod -n my-vm',
    'Create a user-assigned MI':
        'az identity create -g rg-prod -n my-app-identity',
    'Attach user-assigned MI to a Container App':
        'az containerapp identity assign -g rg-prod -n my-app '
        '--user-assigned /subscriptions/.../my-app-identity',
    'Grant MI access to Key Vault secrets':
        'az role assignment create --assignee <MI-principal-id> '
        '--role "Key Vault Secrets User" --scope <kv-resource-id>',
    'Grant MI access to Storage blobs':
        'az role assignment create --assignee <MI-principal-id> '
        '--role "Storage Blob Data Contributor" --scope <storage-resource-id>',
    'Fetch a token from inside a VM or VMSS (IMDS, a fixed link-local address)':
        'curl "http://169.254.169.254/metadata/identity/oauth2/token'
        '?api-version=2018-02-01&resource=https://vault.azure.net" -H Metadata:true',
    'Fetch a token from inside App Service / Functions / Container Apps (NOT IMDS)':
        'curl "$IDENTITY_ENDPOINT?api-version=2019-08-01'
        '&resource=https://vault.azure.net" -H "X-IDENTITY-HEADER: $IDENTITY_HEADER"',
}
for desc, cmd in cli.items():
    print(f'# {desc}')
    print(f'{cmd}\n')


### Where the token actually comes from

Managed identity code never holds a secret; it asks a local endpoint for a token. **Which**
local endpoint depends on where you are running, and this trips people up:

| Host | Endpoint | How you find it |
|------|----------|-----------------|
| VM, Virtual Machine Scale Set | **IMDS** — the Instance Metadata Service at the fixed link-local address `169.254.169.254` | Hard-coded address, plus the `Metadata: true` header |
| App Service, Functions, Container Apps, Azure Automation | A **per-app local token service**, *not* IMDS | The `IDENTITY_ENDPOINT` env var for the URL and `IDENTITY_HEADER` for the header that proves the caller is inside the app (an SSRF mitigation) |
| Azure Arc-enabled server | A local endpoint on **`localhost:40342`** — same path shape as IMDS, different address | `IDENTITY_ENDPOINT` / `IMDS_ENDPOINT`, plus a **challenge token**: the first request 401s with a `WWW-Authenticate` header naming a file only privileged local users can read, and you send its contents back as `Authorization: Basic` |

Hard-coding `169.254.169.254` in an App Service is a classic bug: the address is not
reachable there, and the symptom is a hang or timeout inside `DefaultAzureCredential`
rather than a clear error. Let the SDK figure it out:

```python
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient

# Same code works on your laptop (az login), in CI (env vars / federated creds),
# and in Azure (managed identity, via whichever endpoint this host provides).
credential = DefaultAzureCredential()
client = SecretClient(vault_url='https://my-kv.vault.azure.net', credential=credential)
db_password = client.get_secret('db-password').value
```

With a **user-assigned** identity, a resource can have several identities attached, so the
token request is ambiguous — pass the client ID explicitly
(`DefaultAzureCredential(managed_identity_client_id=...)`, or `client_id=` on
`ManagedIdentityCredential`). System-assigned has exactly one, so nothing to disambiguate.

## Workload Identity Federation (federated credentials)

For workloads *outside* Azure (GitHub Actions, AKS pods, GitLab, other clouds) you can
still get a Microsoft Entra ID token **without** any secret, using **federated credentials**.
The external platform issues its own OIDC token; Entra ID trusts it based on `issuer`,
`subject` and `audience`. Zero long-lived secrets, full audit trail.

```yaml
# Minimal GitHub Actions step that logs in to Azure using federation.
- uses: azure/login@v2
  with:
    client-id:       ${{ secrets.AZURE_CLIENT_ID }}
    tenant-id:       ${{ secrets.AZURE_TENANT_ID }}
    subscription-id: ${{ secrets.AZURE_SUBSCRIPTION_ID }}
    # NOTE: no client-secret! Federation uses the OIDC token GitHub mints for this job.
```

```bash
# On Azure: add a federated credential to the app registration.
az ad app federated-credential create --id <app-id> --parameters '{
  "name": "github-main",
  "issuer": "https://token.actions.githubusercontent.com",
  "subject": "repo:my-org/my-repo:ref:refs/heads/main",
  "audiences": ["api://AzureADTokenExchange"]
}'
```

The `subject` is the security boundary and it is worth reading twice:
`repo:my-org/my-repo:ref:refs/heads/main` trusts *that branch of that repo only*. Widen it
to `repo:my-org/my-repo:*` and any pull request from a fork that can trigger a workflow
gets your Azure credentials.

### Exam tips

- Always prefer **managed identity** over stored secrets.
- For AKS: use **workload identity** (federated, via OIDC) instead of pod-mounted secrets.
- For GitHub Actions / other CI: use **federated credentials** instead of a stored client secret.
- `DefaultAzureCredential` is the recommended credential class for all scenarios.

---
## Summary

| Concept | Key point |
|---------|-----------|
| **App registration** | Client ID + (secret / cert / federated cred), API permissions, exposed scopes, app roles |
| **Delegated vs application** | `scp` claim (user present) vs `roles` claim (app-only) |
| **Consent** | App permissions always need ADMIN consent. Missing consent = valid token, empty `roles`, 403 at the resource. |
| **System-assigned MI** | 1:1 with its resource, auto-deleted, new object ID if the resource is rebuilt |
| **User-assigned MI** | Its own resource, reusable, survives the workload, needs an explicit client ID when several are attached |
| **Token endpoint** | IMDS `169.254.169.254` on VMs and scale sets; `IDENTITY_ENDPOINT` + `IDENTITY_HEADER` on App Service / Functions / Container Apps; `localhost:40342` + a challenge token on Arc servers. Read the env var, never hard-code. |
| **Workload Identity Federation** | Secretless auth for CI/CD, AKS, other clouds. The `subject` is the boundary — pin the branch. |
| **DefaultAzureCredential** | One credential class that works everywhere |


---
## ✅ Self-check

Answer these before moving on. Answers are in the next cell.

1. You inspect an access token and it has a `roles` claim but no `scp` and no `upn`.
   What kind of permission is this, and who consented to it?
2. A daemon successfully gets a token for `api://api-b/.default`, but every call to api-b
   returns 403. The client secret is valid and not expired. What is the most likely cause,
   and where do you look?
3. A VM with a **system-assigned** managed identity has `Key Vault Secrets User` on a
   vault. The VM is destroyed and re-deployed by the same ARM template. Does it still have
   access?
4. Your App Service code calls `http://169.254.169.254/metadata/identity/oauth2/token` and
   hangs. What is wrong?
5. Two user-assigned managed identities are attached to one Container App. Your code calls
   `DefaultAzureCredential()` with no arguments. What happens?
6. A GitHub Actions federated credential has
   `subject: repo:contoso/infra:ref:refs/heads/main`. A contributor opens a pull request
   that changes the workflow to run `az group delete`. Do they get your Azure credentials?
7. Which of these should you use for a nightly job running on an on-prem server that needs
   to write to an Azure storage account: managed identity, client secret, client
   certificate, or workload identity federation?


In [ ]:
answers = """
1. An APPLICATION permission (app role), obtained via the client_credentials grant. There
   is no user in the flow - that is why there is no upn and no scp - so no user could
   possibly have consented. An administrator granted it (admin consent). If you ever see
   BOTH scp and roles, you are looking at a token where a user signed in to an app that
   also holds app roles; the resource should decide which one it honours, and most APIs
   should honour scp only when a user is present.

2. Admin consent was never granted, so the roles claim is empty. Decode the token and look
   at "roles": [] - the token endpoint happily returns 200 for an app with no app roles,
   because issuing a token and being authorised are different things. Fix it with
   `az ad app permission admin-consent --id <app-id>`, then get a NEW token; the old one
   still has the empty claim until it expires.

3. NO. A system-assigned identity dies with its resource. The re-deployed VM gets a brand
   new service principal with a new object ID, and the role assignment still points at the
   old, now-deleted principal. The template has to re-create the role assignment - or, far
   better, use a user-assigned identity that outlives the VM and keeps its object ID.

4. App Service does not expose IMDS. The 169.254.169.254 link-local address is a VM / VMSS
   / Arc thing. On App Service, Functions and Container Apps you read the IDENTITY_ENDPOINT
   environment variable for the URL and send IDENTITY_HEADER as the X-IDENTITY-HEADER
   header. Because nothing answers at that address, the symptom is a timeout rather than an
   error - which is why this one takes an afternoon to find.

5. The token request is ambiguous and fails - the platform cannot guess which of the two
   identities you meant. Pass the identity explicitly:
   DefaultAzureCredential(managed_identity_client_id=<client-id>). Only a system-assigned
   identity is unambiguous, because there can only ever be one.

6. NO - not for the main branch credential. The subject pins ref:refs/heads/main, and a
   pull request's OIDC token has a different subject (repo:contoso/infra:pull_request).
   This is exactly why you do not widen the subject to repo:contoso/infra:*. Note the flip
   side: anyone who can PUSH to main can still use it, so protect the branch too.

7. Managed identities do not exist outside Azure, so that is out. Between the rest,
   workload identity federation if the server's platform can mint an OIDC token that Entra
   ID can trust; otherwise a client CERTIFICATE, which is stronger than a secret and
   rotatable without redeploying code. A client secret is the last resort - and if you use
   one, it belongs in Key Vault with an expiry alert, never in a config file.
"""
print(answers)


**Next lab**: [02 - Secure Networking](../../02-networking/)
